# Графики зависимости признаков от целевой переменной

Точечные графики (scatter plots) для визуализации зависимости различных признаков от целевой переменной — скорости коррозии (`corr_rate_worst_mm_per_year`).

In [ ]:
# Импорты и загрузка данных 
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

# Доступ к проектным модулям
if '../src' not in sys.path:
    sys.path.append('../src')
from database import load_worst_corrosion_by_component as load_data

# Загрузка полного датасета (запускать редко — долго)
DF_full = load_data()

In [ ]:
# Целевая переменная
TARGET = 'corr_rate_worst_mm_per_year'

assert TARGET in DF_full.columns, f'В данных отсутствует {TARGET}'

print(f"Полный датасет: {len(DF_full):,} строк, {len(DF_full.columns)} колонок")
print('Колонки:', sorted(DF_full.columns.tolist()))

In [ ]:
def plot_scatter_vs_target(columns, title, target=None, df=None):
    """Точечные графики: двумерная зависимость переменных от целевой (Y).
    columns — список признаков по оси X, title — название графика, target — целевая переменная (по умолчанию TARGET)."""
    if df is None:
        df = globals().get('DF_loaded', globals().get('DF_full'))
    if target is None:
        target = TARGET
    if target not in df.columns:
        print(f"Целевая переменная '{target}' отсутствует в данных.")
        return
    cols_use = [c for c in columns if c in df.columns and c != target]
    if not cols_use:
        print("Нет подходящих столбцов для построения графика.")
        return
    n = len(cols_use)
    if n == 1:
        fig, ax = plt.subplots(figsize=(8, 5))
        axes = [ax]
    else:
        ncols = min(2, n)
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5 * nrows))
        axes = np.atleast_1d(axes).flatten()
        for a in axes[n:]:
            a.set_visible(False)
    y_label = "Скорость коррозии, мм/год" if "corr_rate" in target else target
    for idx, x_col in enumerate(cols_use):
        ax = axes[idx]
        data = df[[x_col, target]].dropna()
        if data.empty:
            ax.text(0.5, 0.5, "Нет данных", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.scatter(data[x_col], data[target], alpha=0.4, s=15, c="steelblue", edgecolors="none")
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_label if n == 1 else target)
        if n > 1:
            ax.set_title(x_col)
    if n > 1:
        fig.suptitle(title, y=1.02)
    else:
        axes[0].set_title(title)
    plt.tight_layout()
    plt.show()
    if n == 1 and not data.empty:
        print(f"Точек: {len(data):,}")
    elif n > 1:
        print(f"Построено подграфиков: {n}")

In [ ]:
# Фильтр по установкам: новая переменная — быстрый перезапуск без повторной загрузки
# 54 - АВТ-6
# 68 - АВТ-2
# 53 - АВТ-1
# 80 - АВТ-5
# 4 - KK-2
# 51- КК
INSTALLATION_IDS = [4,51,80,53,68,54]  # Список ID установок для фильтрации 


# Фильтрация по списку установок
DF_loaded = DF_full[DF_full['installation_id'].isin(INSTALLATION_IDS)].copy()
DF_loaded = DF_loaded.sort_values('installation_id').reset_index(drop=True)
print(f"Отфильтровано (installation_id in {INSTALLATION_IDS}): {len(DF_loaded):,} строк")

# Только неотрицательная целевая переменная
DF_loaded = DF_loaded[DF_loaded[TARGET] >= 0]
print(f"После отбора строк с {TARGET} >= 0: {len(DF_loaded):,} строк")

# Пустые в числовых признаках = 0; цель не заполняем
num_cols = [c for c in DF_loaded.select_dtypes(include=[np.number]).columns if c != TARGET]
DF_loaded[num_cols] = DF_loaded[num_cols].fillna(0)

print(f"Распределение по установкам:")
print(DF_loaded['installation_id'].value_counts().sort_index())

In [ ]:
# Точечный график: зависимость переменной от целевой (скорость коррозии)
plot_scatter_vs_target(['water_content'], 'Скорость коррозии от содержания воды')
# Точечный график: зависимость переменной от целевой (скорость коррозии)
plot_scatter_vs_target(['h2s_content'], 'Скорость коррозии от содержания воды')
plot_scatter_vs_target(['cross_section_area_mm2'], 'Скорость коррозии от содержания воды')
plot_scatter_vs_target(['stress_corrosion_index'],'')